# Fourier Shell Correlation (FSC) Analysis

Computes and plots FSC curves between pairs of 3-D cryoEM volumes in MRC format.

**Resolution criteria:**
- **FSC = 0.143** — gold-standard half-map criterion (Rosenthal & Henderson 2003)
- **FSC = 0.500** — classic half-bit criterion

---

In [ ]:
import sys
import os

# Add the project root to the path so the fsc package is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsc.fsc_calculator import compute_fsc_from_files
from fsc.fsc_plotter   import plot_fsc, plot_fsc_grid

%matplotlib inline
print("Imports OK")

## 1 · Configure volumes and parameters

Set `MAPS_DIR` to the folder containing your MRC files.  
All unique pairwise combinations are computed automatically.

In [ ]:
import glob
from itertools import combinations
from pathlib import Path

MAPS_DIR   = "maps"          # ← change to the folder containing your MRC files
THRESHOLDS = [0.143, 0.5]
NUM_SHELLS = None             # None → box_size // 2 (auto)
XLIM       = (8.0, 2.0)      # resolution axis limits in Å  (left, right)

# Auto-discover all MRC files and build every unique pair
mrc_files = sorted(glob.glob(f"{MAPS_DIR}/*.mrc"))
PAIRS = [
    (a, b, f"{Path(a).stem}  vs  {Path(b).stem}")
    for a, b in combinations(mrc_files, 2)
]

print(f"Found {len(mrc_files)} volume(s) → {len(PAIRS)} unique pair(s):")
for _, _, label in PAIRS:
    print(f"  {label}")

## 2 · Compute FSC curves

Each pair is processed independently.  The computation is performed entirely in
NumPy via 3-D FFT; for a 288³ box expect ~5–15 seconds per pair.

In [ ]:
results = []

for path1, path2, label in PAIRS:
    print(f"  Computing FSC: {label} … ", end="", flush=True)
    r = compute_fsc_from_files(path1, path2, num_shells=NUM_SHELLS, thresholds=THRESHOLDS)
    r["label"] = label          # override auto-generated label
    results.append(r)
    for thr, res in r["resolutions"].items():
        res_str = f"{res:.2f} Å" if np.isfinite(res) else "N/A"
        print(f"  d{thr:.3f} = {res_str}", end="")
    print()

print("\nDone.")

## 3 · Resolution summary table

In [ ]:
rows = []
for r in results:
    row = {"Pair": r["label"], "Voxel size (Å)": f"{r['voxel_size']:.3f}"}
    for thr, res in r["resolutions"].items():
        col = f"d{thr:.3f} resolution (Å)"
        row[col] = f"{res:.2f}" if np.isfinite(res) else "—"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Pair")
df

## 4 · Overlay plot — all curves on one axes

In [ ]:
fig, ax = plot_fsc(
    results,
    thresholds=THRESHOLDS,
    title="FSC — all pairs",
    figsize=(9, 5),
    show_annotations=False,   # too crowded for multiple curves
    xlim=XLIM,
)
plt.show()

## 5 · Individual panel grid

In [ ]:
fig, axes = plot_fsc_grid(
    results,
    thresholds=THRESHOLDS,
    ncols=2,
    figsize_per_panel=(6, 4),
    suptitle="FSC — individual panels",
    xlim=XLIM,
)
plt.show()

## 6 · Export FSC data to CSV

In [ ]:
import os

OUT_DIR = "fsc_output"
os.makedirs(OUT_DIR, exist_ok=True)

# ── per-pair FSC curve CSVs ───────────────────────────────────────────────────
for r in results:
    safe_label = r["label"].replace(" ", "_").replace("/", "-").replace("  vs  ", "_vs_")
    out_path   = os.path.join(OUT_DIR, f"fsc_{safe_label}.csv")

    # Build one row per resolution threshold as extra columns
    res_cols = {f"d{thr:.3f}_resolution_A": r["resolutions"].get(thr, np.nan)
                for thr in THRESHOLDS}

    df_out = pd.DataFrame({
        "spatial_freq_cyc_per_px": r["shell_centers"],
        "resolution_A":            r["resolution_A"],
        "fsc":                     r["fsc"],
    })
    # Append resolution estimates as constant columns so they travel with the data
    for col, val in res_cols.items():
        df_out[col] = val

    df_out.to_csv(out_path, index=False, float_format="%.6f")
    print(f"Saved curve: {out_path}")

# ── summary CSV — one row per pair ───────────────────────────────────────────
summary_rows = []
for r in results:
    row = {
        "pair":          r["label"],
        "volume_1":      r["path1"],
        "volume_2":      r["path2"],
        "voxel_size_A":  r["voxel_size"],
        "n_shells":      len(r["shell_centers"]),
    }
    for thr, res in r["resolutions"].items():
        row[f"d{thr:.3f}_resolution_A"] = round(res, 3) if np.isfinite(res) else None
    summary_rows.append(row)

summary_path = os.path.join(OUT_DIR, "fsc_summary.csv")
pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
print(f"\nSaved summary: {summary_path}")
pd.DataFrame(summary_rows).set_index("pair")